In [1]:
import pandas as pd
from prophet import Prophet
from dotenv import load_dotenv
import plotly.express as px
import os
import utils
load_dotenv()

/home/nbillard/miniconda3/envs/conda-jedha/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
os.getenv('DATA_STORAGE')

'../../../data/csv/'

In [2]:
df_rte = pd.read_csv(os.getenv('DATA_STORAGE') + 'power_consumption_meteo_2021_2025_utc.csv')
df_rte.head()

,start_date,end_date,value,mean,min,max,median,std,q1,q3
0,2021-01-01 00:00:00,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 01:00:00,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 02:00:00,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 03:00:00,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 04:00:00,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


In [3]:
dfp = utils.convert_df_to_prophet(df_rte)
dfp.head()

,ds,y,mean,min,max,median,std,q1,q3
0,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


--------------------------------------------------------------------

In [ ]:
dataset = dfp.copy()
dataset = dataset[dataset['ds'] < '2025-10-01']
train_period = "30 days"
test_period = "1 day"
period = "53 hours" # to have differents dayofweek and hours
regressors = ['mean']
title = f"train_period: {train_period} - test_period: {test_period} - period: {period} - regressors: {",".join(regressors)}"
train_test_sets = utils.create_train_test(dataset, train_period, test_period, period=period, n_iter=5)

In [31]:
train_test_sets[0][0].head()

,ds,y,mean,min,max,median,std,q1,q3
40866,2025-08-30 23:00:00,37596.0,18.192,14.1,21.7,19.00,1.847982,16.625,19.400
40867,2025-08-31 00:00:00,35898.5,17.744,13.1,20.8,18.50,1.783960,16.525,18.875
40868,2025-08-31 01:00:00,34092.0,17.530,13.2,20.2,18.35,1.782712,16.250,18.700
40869,2025-08-31 02:00:00,32253.0,17.338,13.1,21.1,18.00,1.785987,16.500,18.400
40870,2025-08-31 03:00:00,31275.5,17.170,12.8,20.0,17.80,1.690097,15.875,18.300


In [9]:
results_reg = utils.train_predict(train_test_sets, regressors=regressors)

09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1056.50, Test MAE: 1393.37
Train MAPE: 2.62, Test MAPE: 3.79
--------------------------------------------------
SET 1
train from 2025-08-24 08:00:00 to 2025-09-23 07:00:00
test from 2025-09-23 08:00:00 to 2025-09-24 07:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1015.30, Test MAE: 1841.45
Train MAPE: 2.51, Test MAPE: 3.97
--------------------------------------------------
SET 2
train from 2025-08-26 13:00:00 to 2025-09-25 12:00:00
test from 2025-09-25 13:00:00 to 2025-09-26 12:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:22 - cmdstanpy - INFO - Chain [1] start processing
09:13:22 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1024.03, Test MAE: 1237.90
Train MAPE: 2.52, Test MAPE: 2.50
--------------------------------------------------
SET 3
train from 2025-08-28 18:00:00 to 2025-09-27 17:00:00
test from 2025-09-27 18:00:00 to 2025-09-28 17:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:22 - cmdstanpy - INFO - Chain [1] start processing
09:13:22 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1067.95, Test MAE: 2939.06
Train MAPE: 2.61, Test MAPE: 7.31
--------------------------------------------------
SET 4
train from 2025-08-30 23:00:00 to 2025-09-29 22:00:00
test from 2025-09-29 23:00:00 to 2025-09-30 22:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')
Train MAE: 1160.17, Test MAE: 1970.16
Train MAPE: 2.82, Test MAPE: 4.41
--------------------------------------------------


In [10]:
results = utils.train_predict(train_test_sets)

09:13:27 - cmdstanpy - INFO - Chain [1] start processing
09:13:27 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:27 - cmdstanpy - INFO - Chain [1] start processing
09:13:27 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1060.76, Test MAE: 1881.95
Train MAPE: 2.63, Test MAPE: 5.05
--------------------------------------------------
SET 1
train from 2025-08-24 08:00:00 to 2025-09-23 07:00:00
test from 2025-09-23 08:00:00 to 2025-09-24 07:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:27 - cmdstanpy - INFO - Chain [1] start processing
09:13:27 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1038.13, Test MAE: 1338.96
Train MAPE: 2.56, Test MAPE: 2.87
--------------------------------------------------
SET 2
train from 2025-08-26 13:00:00 to 2025-09-25 12:00:00
test from 2025-09-25 13:00:00 to 2025-09-26 12:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:27 - cmdstanpy - INFO - Chain [1] start processing
09:13:27 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1054.40, Test MAE: 1464.78
Train MAPE: 2.59, Test MAPE: 2.92
--------------------------------------------------
SET 3
train from 2025-08-28 18:00:00 to 2025-09-27 17:00:00
test from 2025-09-27 18:00:00 to 2025-09-28 17:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


09:13:28 - cmdstanpy - INFO - Chain [1] start processing
09:13:28 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1073.17, Test MAE: 2890.09
Train MAPE: 2.63, Test MAPE: 7.19
--------------------------------------------------
SET 4
train from 2025-08-30 23:00:00 to 2025-09-29 22:00:00
test from 2025-09-29 23:00:00 to 2025-09-30 22:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')
Train MAE: 1157.83, Test MAE: 2005.55
Train MAPE: 2.82, Test MAPE: 4.49
--------------------------------------------------


In [33]:
# chart predict
def chart_predictions(df, results, results_reg, title, display_period="30 days"):

    display_delta = pd.Timedelta(display_period)
    print(display_delta)
    last_date = df['ds'].max()
    display_start = last_date - display_delta
    display_mask = df['ds'] > display_start
    print(last_date, display_start)

    fig = px.line(df[display_mask], x='ds', y='y', range_y=[0, None], title=title)

    i = 1
    for res in results:
        train_pred, test_pred = res
        fig.add_trace(px.scatter(test_pred, x='ds', y='yhat').data[0])
        fig.data[i].marker = dict(color='red', size=3)
        i = i+1

    for res in results_reg:
        train_pred, test_pred = res
        fig.add_trace(px.scatter(test_pred, x='ds', y='yhat').data[0])
        fig.data[i].marker = dict(color='green', size=3)
        i = i+1
        
    fig.show()

In [34]:
chart_predictions(dataset, results, results_reg, title, "15 days")

15 days 00:00:00
2025-09-30 23:00:00 2025-09-15 23:00:00
